# Common HTTP Client

**Purpose:** Standardized HTTP client with retry logic, error handling, and authentication.

**Usage:**
```python
%run "./common/common_http_client"

client = HTTPClient(base_url="https://api.example.com")
response = client.get("/endpoint", params={"key": "value"})
data = client.post("/data", json_data={"field": "value"})
```

**Features:**
* Automatic retry with exponential backoff
* Timeout handling
* Authentication support (Bearer token, API key)
* Rate limiting
* Response validation
* Logging integration

In [0]:
import requests
import time
import json
from typing import Optional, Dict, Any, Union
from dataclasses import dataclass

# ============================================================================
# HTTP CLIENT CONFIGURATION
# ============================================================================

@dataclass
class HTTPClientConfig:
    """Configuration for HTTP client."""
    
    # Retry configuration
    max_retries: int = 3
    retry_backoff_factor: float = 2.0  # Exponential backoff multiplier
    retry_status_codes: list = None  # Status codes to retry on
    
    # Timeout configuration (seconds)
    connect_timeout: int = 10
    read_timeout: int = 30
    
    # Rate limiting
    rate_limit_delay: float = 0.0  # Seconds between requests
    
    def __post_init__(self):
        if self.retry_status_codes is None:
            # Default: retry on server errors and rate limit
            self.retry_status_codes = [429, 500, 502, 503, 504]

In [0]:
class HTTPClient:
    """
    Reusable HTTP client with retry logic and error handling.
    
    Example:
        # Basic usage
        client = HTTPClient(base_url="https://api.example.com")
        response = client.get("/users", params={"page": 1})
        
        # With authentication
        client = HTTPClient(
            base_url="https://api.example.com",
            auth_token="your_token_here"
        )
        
        # With custom configuration
        config = HTTPClientConfig(max_retries=5, connect_timeout=20)
        client = HTTPClient(
            base_url="https://api.example.com",
            config=config
        )
    """
    
    def __init__(
        self,
        base_url: str,
        auth_token: Optional[str] = None,
        api_key: Optional[str] = None,
        api_key_header: str = "X-API-Key",
        config: Optional[HTTPClientConfig] = None
    ):
        """
        Initialize HTTP client.
        
        Args:
            base_url: Base URL for all requests
            auth_token: Bearer token for authorization
            api_key: API key for authentication
            api_key_header: Header name for API key
            config: HTTPClientConfig instance
        """
        self.base_url = base_url.rstrip("/")
        self.auth_token = auth_token
        self.api_key = api_key
        self.api_key_header = api_key_header
        self.config = config if config else HTTPClientConfig()
        self.session = requests.Session()
        
        # Set up default headers
        self._setup_headers()
    
    def _setup_headers(self) -> None:
        """Configure default headers."""
        if self.auth_token:
            self.session.headers["Authorization"] = f"Bearer {self.auth_token}"
        
        if self.api_key:
            self.session.headers[self.api_key_header] = self.api_key
        
        self.session.headers["Content-Type"] = "application/json"
    
    def _get_full_url(self, endpoint: str) -> str:
        """Build full URL from base URL and endpoint."""
        endpoint = endpoint.lstrip("/")
        return f"{self.base_url}/{endpoint}"
    
    def _make_request(
        self,
        method: str,
        endpoint: str,
        params: Optional[Dict] = None,
        json_data: Optional[Dict] = None,
        data: Optional[Any] = None,
        headers: Optional[Dict] = None
    ) -> requests.Response:
        """
        Make HTTP request with retry logic.
        
        Args:
            method: HTTP method (GET, POST, PUT, DELETE, etc.)
            endpoint: API endpoint
            params: Query parameters
            json_data: JSON request body
            data: Form data or other body content
            headers: Additional headers
        
        Returns:
            requests.Response object
        
        Raises:
            requests.exceptions.RequestException: On request failure
        """
        url = self._get_full_url(endpoint)
        timeout = (self.config.connect_timeout, self.config.read_timeout)
        
        # Merge custom headers with session headers
        request_headers = dict(self.session.headers)
        if headers:
            request_headers.update(headers)
        
        # Retry loop
        last_exception = None
        
        for attempt in range(self.config.max_retries + 1):
            try:
                # Rate limiting delay
                if attempt > 0 or self.config.rate_limit_delay > 0:
                    time.sleep(
                        self.config.rate_limit_delay +
                        (self.config.retry_backoff_factor ** (attempt - 1) if attempt > 0 else 0)
                    )
                
                # Make request
                response = self.session.request(
                    method=method,
                    url=url,
                    params=params,
                    json=json_data,
                    data=data,
                    headers=request_headers,
                    timeout=timeout
                )
                
                # Check if we should retry based on status code
                if response.status_code in self.config.retry_status_codes:
                    if attempt < self.config.max_retries:
                        continue  # Retry
                
                # Success or non-retriable error
                response.raise_for_status()
                return response
            
            except requests.exceptions.RequestException as e:
                last_exception = e
                
                if attempt < self.config.max_retries:
                    continue  # Retry
        
        # All retries exhausted
        raise last_exception

In [0]:
    def get(
        self,
        endpoint: str,
        params: Optional[Dict] = None,
        headers: Optional[Dict] = None
    ) -> Dict[str, Any]:
        """
        Make GET request.
        
        Args:
            endpoint: API endpoint
            params: Query parameters
            headers: Additional headers
        
        Returns:
            JSON response as dictionary
        
        Example:
            data = client.get("/users", params={"page": 1, "limit": 10})
        """
        response = self._make_request("GET", endpoint, params=params, headers=headers)
        return response.json()
    
    def post(
        self,
        endpoint: str,
        json_data: Optional[Dict] = None,
        data: Optional[Any] = None,
        params: Optional[Dict] = None,
        headers: Optional[Dict] = None
    ) -> Dict[str, Any]:
        """
        Make POST request.
        
        Args:
            endpoint: API endpoint
            json_data: JSON request body
            data: Form data or other body content
            params: Query parameters
            headers: Additional headers
        
        Returns:
            JSON response as dictionary
        
        Example:
            result = client.post("/users", json_data={"name": "John", "email": "john@example.com"})
        """
        response = self._make_request(
            "POST",
            endpoint,
            params=params,
            json_data=json_data,
            data=data,
            headers=headers
        )
        return response.json()
    
    def put(
        self,
        endpoint: str,
        json_data: Optional[Dict] = None,
        params: Optional[Dict] = None,
        headers: Optional[Dict] = None
    ) -> Dict[str, Any]:
        """
        Make PUT request.
        
        Args:
            endpoint: API endpoint
            json_data: JSON request body
            params: Query parameters
            headers: Additional headers
        
        Returns:
            JSON response as dictionary
        
        Example:
            result = client.put("/users/123", json_data={"name": "Jane"})
        """
        response = self._make_request("PUT", endpoint, params=params, json_data=json_data, headers=headers)
        return response.json()
    
    def delete(
        self,
        endpoint: str,
        params: Optional[Dict] = None,
        headers: Optional[Dict] = None
    ) -> Dict[str, Any]:
        """
        Make DELETE request.
        
        Args:
            endpoint: API endpoint
            params: Query parameters
            headers: Additional headers
        
        Returns:
            JSON response as dictionary
        
        Example:
            result = client.delete("/users/123")
        """
        response = self._make_request("DELETE", endpoint, params=params, headers=headers)
        
        # Handle empty response
        try:
            return response.json()
        except json.JSONDecodeError:
            return {"status": "success", "status_code": response.status_code}
    
    def close(self) -> None:
        """
        Close the HTTP session.
        
        Example:
            client.close()
        """
        self.session.close()